In [2]:
from typing import Any

import torch
import torch.nn.functional as F
from torch import nn

device = (
    "mps"
    if torch.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)

## Contrastive Loss
Loss Highlighting L2 normalization and cosine similarity using projection headm

In [3]:
class ContrastiveModel(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, embedding_dim=128):
        super(ContrastiveModel, self).__init__()

        # Base Encoder extracts h
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # Projection Head Maps to the final space where h is applied
        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)

        # L2 Normalization Euclidean
        z_normalized = F.normalize(z, p=2, dim=1)
        return z_normalized

In [4]:
def contrastive_loss(z_i, z_j, temperature=0.5):
    """
    Info NCE Loss
    """
    B = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)

    # Cosine Similarity Matrix
    # Because z is L2 normalized, a simple dot product computes the cosine similarity instantly.
    # Shape will be (2N, 2N)
    sim_matrix = z @ z.T / temperature

    # Mask out the diagonal (a vector's similarity to itself is always 1,
    # we ignore it
    mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim_matrix.masked_fill_(mask, -9e15)

    # Create labels: for z_i[0], the positive match is z_j[0] (which is at index 0 + batch_size)

    labels = torch.arange(B, device=z.device)
    labels = torch.cat([labels + B, labels])

    # Computing Cross Entropy Loss
    loss = F.cross_entropy(sim_matrix, labels)
    return loss

In [5]:
B = 16
embedding_dim = 128
model = ContrastiveModel(input_dim=784, hidden_dim=400, embedding_dim=embedding_dim)

In [6]:
model

ContrastiveModel(
  (encoder): Sequential(
    (0): Linear(in_features=784, out_features=400, bias=True)
    (1): ReLU()
    (2): Linear(in_features=400, out_features=400, bias=True)
    (3): ReLU()
  )
  (projector): Sequential(
    (0): Linear(in_features=400, out_features=400, bias=True)
    (1): ReLU()
    (2): Linear(in_features=400, out_features=128, bias=True)
  )
)

## Contrastive Model by augmentation of 2 views

In [7]:
z_i = torch.rand(B, 784)
z_j = torch.rand(B, 784)

loss = contrastive_loss(z_j, z_j, temperature=0.5)

In [8]:
loss

tensor(0.)

In [9]:
similarity = torch.sum(z_i[0] + z_j[0])
similarity.item()

784.246337890625


No Decoder: Contrastive learning models generally throw away the projection head after training and just use the base encoder to get feature vectors for downstream tasks like classification or clustering.

The Unit Hypersphere: Notice the F.normalize(z, p=2, dim=1) line. Instead of using a KL divergence loss to enforce a Gaussian distribution, contrastive learning manually forces all vectors to have an exact L2 norm (magnitude) of 1.

Instant Distance Math: Because every vector has a length of 1, the complex Mahalanobis distance calculation collapses entirely. You just multiply the vectors together (torch.matmul(z, z.T)), and you instantly have your similarity score.

In the context of the contrastive learning code we discussed (which is heavily based on the famous SimCLR framework), **`h`** and the **projection head** are two distinct parts of the model architecture. They serve completely different purposes during and after training.

Here is the conceptual breakdown of what they are and why this separation exists.

### What is `h`? (The Base Representation)

* **The Output:** `h` is the output vector from your primary neural network (the **base encoder**). If you are processing images, this base encoder might be a ResNet. If you are processing text, it might be a Transformer.
* **The Goal:** `h` is the "holy grail" of the training process. It is a dense, rich representation that captures the fundamental semantic meaning of your input data.
* **Downstream Use:** Once your model is fully trained, **`h` is the only thing you keep.** If you want to build a classifier later, you will pass your data through the encoder to get `h`, and then feed `h` into a simple linear classifier.

### What is the Projection Head? (The Training Buffer)

* **The Structure:** The projection head is just a tiny neural network (usually 2 or 3 linear layers with a ReLU activation in the middle) that sits directly on top of the base encoder. It takes `h` as its input and outputs a new vector, **`z`**.
* **The Target:** The contrastive loss function (pulling similar things together, pushing different things apart) is calculated exclusively on **`z`**, not on `h`.
* **The Disposal:** After the model finishes training, you literally delete the projection head. It is entirely thrown away.

---

### Why do we need a Projection Head?

If `h` is the representation we actually want, why don't we just apply the contrastive loss directly to `h`? Why build a projection head just to throw it away?

It comes down to acting as an **information buffer**.

When you train a contrastive model, you use heavy data augmentations. For an image, you might aggressively crop it, drop all its color (grayscale), and blur it. The contrastive loss forces the network to say, *"These two distorted images are the exact same thing."*

To achieve this, the network is forced to **destroy information**. To make a colorful image and a grayscale image perfectly match in the representation space, the network must learn to completely ignore color.

1. If you apply the loss directly to `h`, `h` becomes completely colorblind. This is bad if your downstream task eventually needs to classify things based on color (like sorting red apples from green apples).
2. By applying the loss to `z`, the **projection head** is forced to become colorblind and absorb the destructive effects of the contrastive loss.
3. Because the projection head acts as a shield, the base encoder is allowed to keep richer, more generalized information inside **`h`**.

In short: the projection head is a sacrificial layer that learns the specific, narrow task of contrastive matching, allowing `h` to remain a highly generalized representation.

Does this clarify why modern representation learning relies so heavily on architectures that you end up partially deleting?

In [ ]:
class LowerLayerPerturbation:
    def __init__(self, model, target_layer, epsilon=0.03):
        self.model = model
        self.target_layer = target_layer
        self.epsilon = epsilon
        self.layer_output = None

        # Register a forward hook to grab the lower layer's features
        self.hook = self.target_layer.register_forward_hook(self.get_features)

    def get_features(self, module, input, output):
        # Save intermediate feature representations
        self.layer_output = output

    def generate_attack(self, x):
        # Force PyTorch to track gradients on the input image
        x = x.clone().detach().requires_grad_(True)

        # Pass input through the network to trigger the hook
        _ = self.model(x)

        # Define an objective on the lower layer features (e.g., maximize variance/norm)
        # This disrupts the foundational feature extraction process
        loss = torch.norm(self.layer_output)

        # Backpropagate to the input image
        loss.backward()

        # Create the adversarial perturbation using the sign of the gradient
        with torch.no_grad():
            perturbation = self.epsilon * x.grad.sign()
            adversarial_x = x + perturbation
            # Keep pixel values valid
            adversarial_x = torch.clamp(adversarial_x, 0.0, 1.0)

        return adversarial_x

    def remove_hook(self):
        # Clean up the hook to free memory
        self.hook.remove()


# --- Usage Example ---
def main():
    # Create a simple dummy model
    model = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),  # Target lower layer
        nn.ReLU(),
        nn.Flatten(),
        nn.Linear(16 * 32 * 32, 10),
    ).eval()

    # Initialize attacker targeting the first layer
    attacker = LowerLayerPerturbation(model, model[0], epsilon=0.02)

    # Generate a random dummy image (Batch size 1, 3 channels, 32x32)
    dummy_image = torch.rand(1, 3, 32, 32)

    # Create the perturbed image
    adv_image = attacker.generate_attack(dummy_image)
    print("Original shape:", dummy_image.shape)
    print("Adversarial shape:", adv_image.shape)

    # Clean up
    attacker.remove_hook()